# Lab 5 — GPU shade mapping with PyCUDA
**Urban question:** how does the shadow pattern of a Philadelphia block evolve between 8:00 and 17:00 — where and when do buildings shade the ground surface?

A digital surface model (DSM) plus the hourly solar position gives one shadow raster per hour. Rendering ten of them as one GIF shows the shadow dynamics of the block: useful for pedestrian heat exposure, building-height inference and urban design.

**Method:** every pixel launches a ray toward the sun (per-pixel ray tracing on the GPU). If any surface along the ray is higher than the sun ray at that distance, the pixel is in shadow.

**How to run:** this notebook is self-contained — it installs PyCUDA, downloads the DSM and regenerates `shadow_analysis.gif` from scratch. Use a Colab **T4 GPU** runtime (Runtime → Change runtime type → T4 GPU) and run all cells top to bottom. There is no randomness; every run produces the same GIF.

*Course lab (AI for Urban Sustainability, UPenn, instr. Xiaojiang Li) — homework: hourly shadow maps 8 am–5 pm combined into a GIF. My solution: Qiwen Bian.*


## 1. Setup
PyCUDA compiles the CUDA kernel on the GPU machine; `gdown` (preinstalled on Colab) fetches the course DSM from Google Drive.


In [ ]:
%pip install -q pycuda

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pycuda.autoinit                     # PyCUDA autoinit: picks the first GPU
import pycuda.driver as cuda
import rasterio as rio
from pycuda.compiler import SourceModule

print("PyCUDA ready. %d device(s) found." % cuda.Device.count())
for ordinal in range(cuda.Device.count()):
    dev = cuda.Device(ordinal)
    print("Device #%d: %s" % (ordinal, dev.name()))


## 2. Data: download and read the DSM
The course provides the DSM tile `row11-col17.tif` (1 m resolution, Pennsylvania StatePlane **feet**, EPSG:2272, Philadelphia). It is downloaded directly from the Lab 05 Drive folder, so no Drive mount or manual upload is needed.


In [ ]:
# Course DSM tile (md5 4f48c024da95b4337dda37804f02c570):
# https://drive.google.com/file/d/1eEdw7mZ9XOGN6DNQMdiYJoDXkERV0e0n/view
!gdown 1eEdw7mZ9XOGN6DNQMdiYJoDXkERV0e0n -O row11-col17.tif

dsm_dataset = rio.open("row11-col17.tif")
dsm_bounds = dsm_dataset.bounds
dsm_img = dsm_dataset.read(1).astype(np.float32)   # the kernel expects float32
height, width = dsm_img.shape
cell_size = 1.0                                    # 1 m per pixel

print("CRS:", dsm_dataset.crs)
print("Bounds:", dsm_bounds)
print("Shape:", dsm_img.shape, "| dtype:", dsm_img.dtype)


## 3. CUDA kernel: one shadow ray per pixel
The kernel converts sun azimuth/elevation to a ray direction. Azimuth starts from **north** and runs **anti-clockwise**, so the row direction is `dy = -cos(azimuth)`. Each thread walks the ray away from its pixel in 1 m steps, up to 2000 m; the first surface that rises above the ray puts the pixel in shadow. Threads are launched in 16×16 blocks.


In [ ]:
kernel_shadow = """
#define PI 3.1415926
__global__ void calculate_shadow(float *dsm, bool *shadow, int width, int height,
                                 float cell_size, float sun_azimuth, float sun_elevation)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    // Check if the thread is within the bounds of the image
    if (x >= width || y >= height)
        return;

    int idx = y * width + x;

    // Convert sun azimuth and elevation to radians
    float azimuth_rad = sun_azimuth * 3.14159 / 180.0;
    float elevation_rad = sun_elevation * 3.14159 / 180.0;

    // Direction of the shadow ray based on sun azimuth
    float dx = sin(azimuth_rad);
    float dy = -cos(azimuth_rad); // because the azimuth starts from the north
    float dz = tan(elevation_rad);

    // Initial height at the current pixel
    float initial_height = dsm[idx];
    bool in_shadow = false;

    // Trace the shadow ray
    for (float t = cell_size; t < 2000.0f; t += cell_size) { // Limit tracing distance
        int x_offset = x + int(dx * t / cell_size);
        int y_offset = y + int(dy * t / cell_size);

        if (x_offset < 0 || x_offset >= width || y_offset < 0 || y_offset >= height)
            break; // Out of bounds

        int offset_idx = y_offset * width + x_offset;

        // Calculate height along the ray
        float height_along_ray = initial_height + dz * t;

        // Check if there is a higher point along the path
        if (dsm[offset_idx] > height_along_ray) {
            in_shadow = true;
            break;
        }
    }

    // Store shadow result: 1 for shadow, 0 for no shadow
    shadow[idx] = in_shadow;
}
"""
mod = SourceModule(kernel_shadow)
calculate_shadows = mod.get_function("calculate_shadow")

block_size = (16, 16, 1)
grid_size = (int(np.ceil(width / block_size[0])), int(np.ceil(height / block_size[1])), 1)
print("Grid:", grid_size, "x block", block_size)


## 4. Hourly solar positions
The DSM centre is projected from EPSG:2272 to WGS84, then the sun elevation and azimuth for each hour 8:00–17:00 are read from the [NOAA solar calculator](https://gml.noaa.gov/grad/solcalc/azel.html) for that location on the study day.


In [ ]:
from pyproj import Transformer

# DSM centre
center_x = (dsm_bounds.left + dsm_bounds.right) / 2
center_y = (dsm_bounds.bottom + dsm_bounds.top) / 2

# EPSG:2272 -> WGS84
transformer = Transformer.from_crs("EPSG:2272", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(center_x, center_y)
print(f"Latitude: {lat}")
print(f"Longitude: {lon}")


In [ ]:
# (hour, sun_elevation, sun_azimuth) from the NOAA solar calculator
solar_pos_lst = [
    (8, 12.89, 100.72),
    (9, 23.89, 111.46),
    (10, 34.04, 124.11),
    (11, 42.58, 139.89),
    (12, 48.38, 159.70),
    (13, 50.15, 182.62),
    (14, 47.35, 205.07),
    (15, 40.80, 223.93),
    (16, 31.79, 238.86),
    (17, 21.38, 250.96),
]


## 5. Shadow map per hour
For each hour the DSM is copied to the GPU once, the kernel traces every pixel's shadow ray in parallel, and the result comes back as a boolean raster that is saved as one PNG.


In [ ]:
os.makedirs("shadow_maps", exist_ok=True)

for hour, sun_elevation, sun_azimuth in solar_pos_lst:
    d_dsm = cuda.mem_alloc(dsm_img.nbytes)
    d_shadow = cuda.mem_alloc(dsm_img.nbytes)
    cuda.memcpy_htod(d_dsm, dsm_img)

    shadow_result = np.zeros_like(dsm_img, dtype=bool)

    calculate_shadows(
        d_dsm, d_shadow,
        np.int32(width), np.int32(height),
        np.float32(cell_size), np.float32(sun_azimuth), np.float32(sun_elevation),
        block=block_size, grid=grid_size,
    )
    cuda.memcpy_dtoh(shadow_result, d_shadow)

    plt.figure(figsize=(8, 8))
    plt.imshow(shadow_result, cmap="gray")
    plt.title(f"Shadow Distribution - {hour}:00")
    plt.axis("off")
    plt.savefig(f"shadow_maps/shadow_{hour:02d}.png", bbox_inches="tight")
    plt.close()

    d_dsm.free()
    d_shadow.free()

print("Saved:", sorted(os.listdir("shadow_maps")))


## 6. Combine the hourly maps into a GIF


In [ ]:
from PIL import Image

files = sorted(glob.glob("shadow_maps/shadow_*.png"))
images = [Image.open(f) for f in files]

images[0].save(
    "shadow_analysis.gif",
    save_all=True,
    append_images=images[1:],
    duration=500,   # 0.5 s per frame
    loop=0,
)
print("Frames:", len(images))


In [ ]:
from IPython.display import Image as IPyImage, display

display(IPyImage(filename="shadow_analysis.gif"))


## 7. Reading the result
- **8:00** — sun low in the east-south-east (elevation 12.9°, azimuth 100.7°): the longest shadows of the day, cast toward west-north-west.
- **13:00** — highest sun (elevation 50.2°): the shadow footprint is smallest, roughly limited to the buildings' own footprints and a narrow fringe.
- **17:00** — sun in the west-south-west (elevation 21.4°, azimuth 251.0°): shadows stretch back toward east-north-east and the shaded share of the block rises again.
- Shadow area shrinks toward midday and grows toward both ends of the day, but is *not* symmetric about noon — the morning/evening asymmetry follows the sun path for the study day.

## Reproducibility
- Runtime: Colab **T4 GPU**; PyCUDA is installed by the notebook (`%pip install -q pycuda`).
- Input: `row11-col17.tif` auto-downloaded via `gdown` (md5 `4f48c024da95b4337dda37804f02c570`), so the notebook has no other dependencies on Drive or local files.
- Deterministic: no random numbers, fixed solar table → identical `shadow_analysis.gif` on every run.
- `shadow_maps/*.png` are intermediate per-hour frames; the final deliverable is the GIF.

*Credits: course Lab 05 notebook (Xiaojiang Li, UPenn); solar positions — NOAA Global Monitoring Laboratory solar calculator.*
